In [1]:
import pandas as pd
from datetime import datetime
from bs4 import BeautifulSoup
import requests
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
import time
import datetime
import re
import os
from unidecode import unidecode
pd.options.display.max_columns = 0
pd.options.display.max_rows = 30
pd.options.display.max_colwidth = 200

In [2]:
def flatten_json(y):
    out = {}
    def flatten(x, name=''):
        if type(x) is dict:
            for a in x:
                flatten(x[a], name + a + '_')
        elif type(x) is list:
            i = 0
            for a in x:
                flatten(a, name + str(i) + '_')
                i += 1
        else:
            out[name[:-1]] = x
 
    flatten(y)
    return out

In [3]:
today = datetime.date.today()

In [4]:
if len(str(today.month)) == 2:
    mon = str(today.month)
else:
    mon = '0'+str(today.month)

if len(str(today.day)) == 2:
    day = str(today.day)
else:
    day = '0'+str(today.day)
url = 'https://api3.natst.at/6843-c01eff/games/NBA/' + str(today.year) + '-' + mon + '-' + day
games = requests.get(url).json()
teams = []

In [5]:
if 'Historical_Data.csv' in os.listdir():
    hd = pd.read_csv('Historical_Data.csv')

C:\Users\jtmck\AppData\Local\Temp\ipykernel_118136\3651976194.py:2: DtypeWarning: Columns (4,15,25,26,28,29,62,63,64) have mixed types. Specify dtype option on import or set low_memory=False.
  hd = pd.read_csv('Historical_Data.csv')


In [6]:
hd['date_string'].nunique()

2159

In [7]:
flag = False

In [8]:
flag = True

In [10]:
"JAKE"

'JAKE'

In [9]:
count = 0
today = datetime.date.today()
if 'Skip_Dates.csv' in os.listdir():
    sd_df = pd.read_csv('Skip_Dates.csv')
else:
    sd_df = pd.DataFrame([{'dates':''}])
if 'Historical_Data.csv' in os.listdir():
    hd = pd.read_csv('Historical_Data.csv')
else:
    hd = pd.DataFrame()
    hd['date_string'] = ''
starting_dates = list(hd['date_string'].unique())
for x in range(0,3650):
    if count < 480:
        pull_date = today - datetime.timedelta(days=x)
        if len(str(pull_date.month)) == 2:
            mon = str(pull_date.month)
        else:
            mon = '0'+str(pull_date.month)
        
        if len(str(pull_date.day)) == 2:
            day = str(pull_date.day)
        else:
            day = '0'+str(pull_date.day)
        date_string = str(pull_date.year) + '-' + mon + '-' + day
        print(date_string)
        #time.sleep(1)
        #try:
        url = 'https://api3.natst.at/6843-c01eff/playerperfs/nba/' + date_string
        if date_string not in starting_dates and date_string not in list(sd_df.dates.unique()):
            print(date_string)
            count += 1
            lines = requests.get(url).json()
            output_data = []
            if 'performances' in list(lines.keys()):
                print(date_string)
                for i in list(lines['performances'].keys()):
                    output_data.append(flatten_json(lines['performances'][i]))
                while lines['meta']['page'] != lines['meta']['pages-total']:
                    count += 1
                    print(lines['meta']['page'],lines['meta']['pages-total'])
                    print(lines['meta'])
                    if 'page-next' in list(lines['meta'].keys()):
                        lines = requests.get(lines['meta']['page-next']).json()
                        for i in list(lines['performances'].keys()):
                            output_data.append(flatten_json(lines['performances'][i]))
                    else:
                        break
                    print(i)
                df = pd.DataFrame(output_data)
                df['date_string'] = date_string
                if flag:
                    df_read = pd.read_csv('Historical_Data.csv')
                else:
                    df_read = pd.DataFrame()
                    flag = True
                df = pd.concat([df,df_read])
                df.to_csv('Historical_Data.csv', index = False)
            else:
                if 'Skip_Dates.csv' in os.listdir():
                    sd_df = pd.read_csv('Skip_Dates.csv')
                    skip_dates = list(sd_df.dates.unique()) + [date_string]
                else:
                    skip_dates = [date_string]
                sd_df = pd.DataFrame(skip_dates)
                sd_df = sd_df.rename(columns={0:'dates'})
                sd_df.to_csv('Skip_Dates.csv')
        #except:
        #    pass
    else:
        time.sleep(3600)
        count = 0

C:\Users\jtmck\AppData\Local\Temp\ipykernel_118136\817620865.py:8: DtypeWarning: Columns (4,15,25,26,28,29,62,63,64) have mixed types. Specify dtype option on import or set low_memory=False.
  hd = pd.read_csv('Historical_Data.csv')


2025-11-14
2025-11-14
2025-11-13
2025-11-13
2025-11-13


C:\Users\jtmck\AppData\Local\Temp\ipykernel_118136\817620865.py:53: DtypeWarning: Columns (4,15,25,26,28,29,62,63,64) have mixed types. Specify dtype option on import or set low_memory=False.
  df_read = pd.read_csv('Historical_Data.csv')


2025-11-12
2025-11-12
2025-11-12
1 3
{'processing-time': '0.24109101295471 sec', 'processed-at': '2025-11-14 17:03:06', 'timezone': 'America/New_York', 'ip': '136.62.52.53', 'ip-last24': '1 (136.62)', 'queryid': 'v35.452100', 'results-max': '100', 'results-total': '282', 'page': '1', 'pages-total': '3', 'page-next': 'https://api3.natst.at/6843-c01eff/playerperfs/nba/2025-11-12/100', 'stat-glossary': 'https://api3.natst.at/6843-c01eff/glossary/nba', 'codes-teams': 'https://api3.natst.at/6843-c01eff/teamcodes/nba', 'codes-leagues': 'https://api3.natst.at/6843-c01eff/leaguecodes/nba', 'api': 'National Statistical API', 'version': '3.65 (2025.11.05)', 'documentation': 'https://natstat.com/api-v3/docs', 'support': 'https://natstat.com/support', 'report_errors': 'https://natstat.com/nba', 'note': 'All times Eastern.'}
performance_12466471
2 3
{'processing-time': '0.18139600753784 sec', 'processed-at': '2025-11-14 17:03:07', 'timezone': 'America/New_York', 'ip': '136.62.52.53', 'ip-last24': '

C:\Users\jtmck\AppData\Local\Temp\ipykernel_118136\817620865.py:53: DtypeWarning: Columns (4,24,57,58,59,61,62,63,64) have mixed types. Specify dtype option on import or set low_memory=False.
  df_read = pd.read_csv('Historical_Data.csv')


...

2025-05-22
2025-05-22
2025-05-22


C:\Users\jtmck\AppData\Local\Temp\ipykernel_118136\817620865.py:53: DtypeWarning: Columns (4,15,16,26,27,29,30,63,64) have mixed types. Specify dtype option on import or set low_memory=False.
  df_read = pd.read_csv('Historical_Data.csv')


2025-05-21
2025-05-21
2025-05-21


C:\Users\jtmck\AppData\Local\Temp\ipykernel_118136\817620865.py:53: DtypeWarning: Columns (4,15,16,26,27,29,30,63,64) have mixed types. Specify dtype option on import or set low_memory=False.
  df_read = pd.read_csv('Historical_Data.csv')


2025-05-20
2025-05-19
2025-05-18
2025-05-17
2025-05-16
2025-05-15
2025-05-14
2025-05-13
2025-05-12
2025-05-11
2025-05-10
2025-05-09
2025-05-08
2025-05-07
2025-05-06
2025-05-05
2025-05-04
2025-05-03
2025-05-02
2025-05-01
2025-04-30
2025-04-29
2025-04-28
2025-04-27
2025-04-26
2025-04-25
2025-04-24
2025-04-23
2025-04-22
2025-04-21
2025-04-20
2025-04-19
2025-04-18
2025-04-17
2025-04-16
2025-04-15
2025-04-14
2025-04-13
2025-04-12
2025-04-11
2025-04-10
2025-04-09
2025-04-08
2025-04-07
2025-04-06
2025-04-05
2025-04-04
2025-04-03
2025-04-02
2025-04-01
2025-03-31
2025-03-30
2025-03-29
2025-03-28
2025-03-27
2025-03-26
2025-03-25
2025-03-24
2025-03-23
2025-03-22
2025-03-21
2025-03-20
2025-03-19
2025-03-18
2025-03-17
2025-03-16
2025-03-15
2025-03-14
2025-03-13
2025-03-12
2025-03-11
2025-03-10
2025-03-09
2025-03-08
2025-03-07
2025-03-06
2025-03-05
2025-03-04
2025-03-03
2025-03-02
2025-03-01
2025-02-28
2025-02-27
2025-02-26
2025-02-25
2025-02-24
2025-02-23
2025-02-22
2025-02-21
2025-02-20
2025-02-19

In [ ]:
print("JAKE")

In [ ]:
date_string = '2024-12-30'
url = 'https://api3.natst.at/6843-c01eff/playerperfs/nba/' + date_string
lines = requests.get(url).json()

In [ ]:
output_data = []
for i in list(lines['performances'].keys()):
    output_data.append(flatten_json(lines['performances'][i]))
while lines['meta']['page'] != lines['meta']['pages-total']:
    lines = requests.get(lines['meta']['page-next']).json()
    for i in list(lines['performances'].keys()):
        output_data.append(flatten_json(lines['performances'][i]))
    print(i)

In [ ]:
pd.DataFrame(output_data)

In [ ]:
lines.keys()

In [ ]:
lines['meta']

In [ ]:
url = 'https://api3.natst.at/6843-c01eff/playerperfs/NBA/'
games = requests.get(url).json()

In [ ]:
len(list(games['performances'].keys()))

In [ ]:
games['performances'].keys()

In [ ]:
pd.DataFrame(games['performances']['performance_12143986'])

In [ ]:
games['performances']['performance_12143986']

In [ ]:
flatten_json(games['performances']['performance_12143986'])

In [ ]:
def flatten_json(y):
    out = {}
    def flatten(x, name=''):
        if type(x) is dict:
            for a in x:
                flatten(x[a], name + a + '_')
        elif type(x) is list:
            i = 0
            for a in x:
                flatten(a, name + str(i) + '_')
                i += 1
        else:
            out[name[:-1]] = x
 
    flatten(y)
    return out

In [ ]:
def parse_stat_line(stat_line):
    